# Topic 5 그림 코드 모음 — 강의용 Colab 데모

**확률통계 · Topic 5 · Entropy and KL Divergence**

슬라이드 [T05_slides_v2.md](slides/T05_slides_v2.md) 에 쓴 그림 3개를 만드는 코드를 모았다.
슬라이드는 고정된 PNG지만, 이 노트북은 **강의 중 숫자를 바꿔가며 즉석에서 다시 그려볼 수 있다.**

| 그림 | 슬라이드 | 원본 스크립트 |
|---|---|---|
| ① gzip 압축 실험 — 같은 길이인데 결과가 377배 차이 | 7쪽 `[S]` gzip으로 압축해 본다 | `figs_src_v2/t05v2_gzip.py` |
| ② 놀라움 곡선 + Bernoulli 엔트로피 | 10쪽 `[C]` 놀라움을 재는 자 | `figs_src_v2/t05v2_entropy.py` |
| ③ cross-entropy = 바닥 + 낭비 | 19쪽 `[C]` 비용 = 바닥 + 낭비 | `figs_src_v2/t05v2_crossent.py` |

⚠️ 실제 슬라이드 그림은 Windows 로컬에서 `Malgun Gothic`으로 렌더했다. 이 노트북은 Colab(Linux)이라
나눔고딕을 대신 설치해서 쓴다 — 글꼴만 다르고 배치·색·수치는 슬라이드와 동일하다.

**맨 위 설정 셀을 한 번 실행한 뒤, 그림 셀은 순서와 상관없이 원하는 것만 실행하면 된다.**

In [ ]:
# 설정: 한글 글꼴 + 스타일 (맨 처음 한 번만 실행)
# Colab은 Linux라 한글 글꼴이 기본으로 없다. 나눔고딕을 설치해 등록한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import gzip

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

try:
    for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(f)
    KOREAN_FONT = "NanumGothic"
except Exception:
    KOREAN_FONT = "DejaVu Sans"  # 설치 실패 시 - 한글은 깨지지만 그림은 그려진다
    print("나눔고딕 설치/등록 실패 - DejaVu Sans로 대신한다 (한글이 깨질 수 있음)")

# 슬라이드 테마와 같은 색 (assets/deckstyle.py 와 동일)
C = {
    "accent": "#3b4fd8", "teal": "#0d9488", "orange": "#d97d17",
    "purple": "#8b5cf6", "pink": "#d9457f", "ink": "#0f172a",
    "body": "#4b5768", "muted": "#97a3b6", "line": "#e6eaf1", "soft": "#f7f9fc",
}
CYCLE = [C["accent"], C["teal"], C["orange"], C["purple"], C["pink"], C["muted"]]

mpl.rcParams.update({
    "font.family": KOREAN_FONT,
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 12.5,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": C["line"],
    "grid.linewidth": 1.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d7dce6",
    "figure.dpi": 110,
})

## ① gzip 압축 실험 — 같은 길이인데 결과가 다르다

세 텍스트를 만든다. **길이(50,000자)와 문자 집합(ASCII a~z)을 똑같이** 맞추고, 확률분포만 바꾼다.

| | 만드는 법 | 엔트로피 |
|---|---|---|
| ① | 같은 글자만 반복 `aaaa...` | 0 bits/char |
| ② | 영어 글자 빈도로 i.i.d. 추출 (e 12.7%, z 0.07%) | 약 4.18 bits/char |
| ③ | 26글자 균등 i.i.d. | 4.70 bits/char |

`gzip.compress(..., 9)` 로 압축한 바이트 수를 이론 하한 $N \times H / 8$ 과 나란히 본다.
강의 중엔 `N` 을 바꾸거나 `FREQ` 를 다른 언어 빈도로 바꿔 다시 돌려볼 수 있다.

⚠️ ②를 "영어 문장 반복"으로 만들면 gzip이 반복 패턴을 잡아 버린다. 반드시 i.i.d. 난수열이어야
분포 하나만 남는다.

In [ ]:
N = 50_000
rng = np.random.default_rng(20260302)          # 개강일 시드 (전 Topic 공통)
letters = np.array(list("abcdefghijklmnopqrstuvwxyz"))

# 영어 글자 상대 빈도 (a~z, %)
FREQ = np.array([8.17, 1.49, 2.78, 4.25, 12.70, 2.23, 2.02, 6.09, 6.97, 0.15,
                 0.77, 4.03, 2.41, 6.75, 7.51, 1.93, 0.10, 5.99, 6.33, 9.06,
                 2.76, 0.98, 2.36, 0.15, 1.97, 0.07])
FREQ = FREQ / FREQ.sum()
UNIF = np.full(26, 1 / 26)


def entropy_bits(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


texts = {
    "① 같은 글자 반복": "a" * N,
    "② 영어 글자 빈도": "".join(rng.choice(letters, size=N, p=FREQ)),
    "③ 26글자 균등": "".join(rng.choice(letters, size=N, p=UNIF)),
}
ents = [0.0, entropy_bits(FREQ), entropy_bits(UNIF)]
sizes = [len(gzip.compress(t.encode("ascii"), 9)) for t in texts.values()]
bounds = [N * h / 8 for h in ents]              # 이론 하한 (bytes)

print(f"{'텍스트':<16}{'맨앞 40자':<44}{'H':>10}{'하한(B)':>12}{'gzip(B)':>12}{'gzip bit/char':>16}")
for name, h, sz, lb, t in zip(texts, ents, sizes, bounds, texts.values()):
    print(f"{name:<16}{t[:40]:<44}{h:>10.4f}{lb:>12,.0f}{sz:>12,}{sz * 8 / N:>16.4f}")
print(f"\n압축 크기 비율  ③ / ① = {sizes[2] / sizes[0]:.0f}배")

# ── 그림: 원본 t05v2_gzip.py 와 동일 배치 ──────────────────────────────
fig, ax = plt.subplots(figsize=(11.6, 3.5))

y = np.arange(3)[::-1]
names = list(texts.keys())[::-1]
sz_r, lb_r, en_r = sizes[::-1], bounds[::-1], ents[::-1]
cols = [C["accent"], C["teal"], C["orange"]]

ax.barh(y, sz_r, height=0.5, color=cols, alpha=0.88, label="실제 gzip 크기")
for i, (s, lb) in enumerate(zip(sz_r, lb_r)):
    if lb > 500:                                # 하한선 (①은 0에 가까워 생략)
        ax.plot([lb, lb], [y[i] - 0.3, y[i] + 0.3], color=C["ink"],
                linewidth=2.4, zorder=5)
    ax.text(s + 700, y[i], f"{s:,} bytes", va="center", fontsize=13.5,
            fontweight="bold", color=C["ink"])

ax.plot([], [], color=C["ink"], linewidth=2.4, label=r"이론 하한 ($N \times H / 8$)")
ax.set_yticks(y)
ax.set_yticklabels([f"{n}\n$H$ = {h:.2f} bits/char" for n, h in zip(names, en_r)])
ax.set_xlabel("압축 후 크기 (bytes) — 원본은 셋 다 50,000자")
ax.set_xlim(0, 40_000)
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## ② 놀라움 곡선 + Bernoulli 엔트로피

왼쪽은 self-information $-\log_2 p$ — 드문 일일수록 놀랍다. 오른쪽은 Bernoulli 엔트로피 $H(p)$
— 균등(0.5)에서 최대 1 bit, 양 끝에서 0. 두 점(`동전 앞면`, `주사위 6` / `치우친 동전`)의
확률을 바꿔가며 "왜 로그인가"를 눈으로 보여줄 수 있다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.5))

# ── 왼쪽: -log2 p ──────────────────────────────────────────
p = np.linspace(0.01, 1.0, 400)
ax1.plot(p, -np.log2(p), color=C["accent"], linewidth=2.4)
for pv, name, col in [(0.5, "동전 앞면", C["teal"]), (1 / 6, "주사위 6", C["orange"])]:
    iv = -np.log2(pv)
    ax1.plot(pv, iv, "o", color=col, markersize=9, zorder=5)
    ax1.annotate(f"{name}\n{iv:.2f} bits", xy=(pv, iv), xytext=(pv + 0.13, iv + 0.9),
                 fontsize=12.5, fontweight="bold", color=col,
                 arrowprops=dict(arrowstyle="-|>", color=col, linewidth=1.4))
ax1.set_title(r"놀라움  $I(x) = -\log_2 p(x)$", color=C["ink"])
ax1.set_xlabel("확률 p")
ax1.set_ylabel("정보량 (bits)")
ax1.set_xlim(0, 1.02)
ax1.set_ylim(0, 6.8)

# ── 오른쪽: Bernoulli 엔트로피 ─────────────────────────────
q = np.linspace(0.001, 0.999, 500)
H = -(q * np.log2(q) + (1 - q) * np.log2(1 - q))
ax2.plot(q, H, color=C["purple"], linewidth=2.4)
ax2.plot(0.5, 1.0, "o", color=C["ink"], markersize=9, zorder=5)
ax2.annotate("p = 0.5 에서 최대  1 bit", xy=(0.5, 1.0), xytext=(0.5, 0.62),
             ha="center", fontsize=12.5, fontweight="bold", color=C["ink"],
             arrowprops=dict(arrowstyle="-|>", color=C["ink"], linewidth=1.4))
h09 = -(0.9 * np.log2(0.9) + 0.1 * np.log2(0.1))
ax2.plot(0.9, h09, "o", color=C["orange"], markersize=8, zorder=5)
ax2.text(0.88, h09 + 0.09, f"치우친 동전\n{h09:.3f} bits", ha="right",
         fontsize=12, fontweight="bold", color=C["orange"])
ax2.set_title(r"Bernoulli 엔트로피  $H(p)$", color=C["ink"])
ax2.set_xlabel("앞면 확률 p")
ax2.set_ylabel("엔트로피 (bits)")
ax2.set_ylim(0, 1.22)

print(f"H(0.5)=1.000 · H(0.9)={h09:.4f} · -log2(1/6)={-np.log2(1 / 6):.4f}")
print(f"H(주사위)=log2(6)={np.log2(6):.4f} · H(4지선다)=2.0")
plt.tight_layout()
plt.show()

## ③ cross-entropy = 바닥 + 낭비

왼쪽: 정답에 준 확률 $q$ 가 낮을수록 손실 $-\log_2 q$ 가 폭발한다. 오른쪽: 예측 $q$ 를 정답
$p=(0.7,\,0.2,\,0.1)$ 쪽으로 옮기면 $H(p,q)$ 가 $H(p)$ 라는 **바닥**까지만 내려가고, 그 위에
남는 것이 $D_{KL}(p\,\|\,q)$ 다. `p` 를 바꿔 다른 정답 분포로 실험해 볼 수 있다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.5))

# ── 왼쪽: 정답에 준 확률에 대한 벌점 ───────────────────────
q = np.linspace(0.01, 1.0, 400)
ax1.plot(q, -np.log2(q), color=C["accent"], linewidth=2.4)
for qv, col in [(0.9, C["teal"]), (0.5, C["orange"]), (0.1, C["pink"])]:
    ax1.plot(qv, -np.log2(qv), "o", color=col, markersize=8, zorder=5)
    ax1.text(qv, -np.log2(qv) + 0.32, f"{-np.log2(qv):.2f}", ha="center",
             fontsize=12.5, fontweight="bold", color=col)
ax1.set_title("정답에 준 확률이 낮을수록 벌점이 폭발한다", color=C["ink"], fontsize=14)
ax1.set_xlabel("모델이 정답에 준 확률 q")
ax1.set_ylabel(r"손실  $-\log_2 q$ (bits)")
ax1.set_xlim(0, 1.02)
ax1.set_ylim(0, 6.8)

# ── 오른쪽: H(p,q) = H(p) + D_KL 분해 ──────────────────────
p = np.array([0.7, 0.2, 0.1])
Hp = float(-(p * np.log2(p)).sum())

alphas = np.linspace(0.0, 1.0, 101)            # 균등 -> p 로 이동
unif = np.full(3, 1 / 3)
xent, kl = [], []
for a in alphas:
    qq = (1 - a) * unif + a * p
    ce = float(-(p * np.log2(qq)).sum())
    xent.append(ce)
    kl.append(ce - Hp)

ax2.fill_between(alphas, Hp, xent, color=C["orange"], alpha=0.22,
                 label=r"$D_{KL}(p\,\|\,q)$  — 낭비")
ax2.fill_between(alphas, 0, Hp, color=C["accent"], alpha=0.16,
                 label=r"$H(p)$  — 어쩔 수 없는 바닥")
ax2.plot(alphas, xent, color=C["orange"], linewidth=2.4,
         label=r"$H(p, q)$  — 내가 치르는 비용")
ax2.axhline(Hp, color=C["accent"], linewidth=2.0, linestyle="--")
ax2.set_title("예측을 정답에 맞출수록 낭비만 줄어든다", color=C["ink"], fontsize=14)
ax2.set_xlabel("예측 q 가 정답 p 에 가까워지는 정도")
ax2.set_ylabel("bits")
ax2.set_ylim(0, 1.85)
ax2.legend(loc="upper right", fontsize=11)

print(f"p = (0.7, 0.2, 0.1)")
print(f"H(p)      = {Hp:.4f} bits")
print(f"H(p, 균등) = {xent[0]:.4f} bits")
print(f"D_KL(p||q) = {kl[0]:.4f} bits   (q = 균등)")
kl_rev = float(-(unif * np.log2(p)).sum()) - float(-(unif * np.log2(unif)).sum())
print(f"D_KL(q||p) = {kl_rev:.4f} bits   -> 비대칭")
plt.tight_layout()
plt.show()

---
이 노트북은 채점 대상이 아니다. 슬라이드 그림을 재생성하는 [figs_src_v2/](figs_src_v2/) 스크립트가
원본이며, 슬라이드 PNG를 바꾸려면 그쪽을 고치고 다시 실행해야 한다 — 이 노트북은 강의 중
라이브 데모·질의응답용 사본이다. 텍스트로 직접 엔트로피·Huffman·KL 을 재는 실습은
[lab/T05_lab.ipynb](lab/T05_lab.ipynb) 에 있다.